In [3]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.linear_model import LinearRegression
import plotly.express as px

def simulate_gambler_ruin(C0, C_max, p, T_max):
    capital = C0
    history = [capital]
    for t in range(T_max):
        if capital == 0 or capital == C_max:
            break
        if np.random.rand() < p:
            capital += 1
        else:
            capital -= 1
        history.append(capital)
    return history, capital

def run_simulations(N, C0, C_max, p, T_max):
    ruin_count = 0
    success_count = 0
    ruin_times = []
    success_times = []
    all_histories = []
    for _ in range(N):
        history, final = simulate_gambler_ruin(C0, C_max, p, T_max)
        all_histories.append(history)
        if final == 0:
            ruin_count += 1
            ruin_times.append(len(history)-1)
        elif final == C_max:
            success_count += 1
            success_times.append(len(history)-1)
    prob_ruin = ruin_count / N
    prob_success = success_count / N
    return {
        'prob_ruin': prob_ruin,
        'prob_success': prob_success,
        'ruin_times': ruin_times,
        'success_times': success_times,
        'all_histories': all_histories
    }

def theoretical_ruin_prob(C0, C_max, p):
    if p == 0.5:
        return 1 - C0 / C_max
    else:
        q = 1 - p
        r = q / p
        return (r**C0 - r**C_max) / (1 - r**C_max)

# Parameters
C0 = 10
C_max = 20
p = 0.45
T_max = 1000
N = 1000

results = run_simulations(N, C0, C_max, p, T_max)

# 2a. Probabilità di rovina e successo
print("Probabilità di rovina:", results['prob_ruin'])
print("Probabilità di successo:", results['prob_success'])

# 2b. Andamento capitale
df_hist = []
for i, h in enumerate(results['all_histories'][:10]):
    df_hist.extend({'Simulazione': i, 'Tempo': t, 'Capitale': c} for t, c in enumerate(h))
fig = px.line(
    pd.DataFrame(df_hist),
    x='Tempo', y='Capitale', color='Simulazione',
    title='Andamento del capitale (prime 10 simulazioni)'
)
fig.show()

# 3a. Cambiando T_max
for T in [100, 500, 1000]:
    res = run_simulations(N, C0, C_max, p, T)
    print(f"T_max={T}: Probabilità di rovina={res['prob_ruin']:.3f}")

# 3b. Tempo atteso per rovina
if results['ruin_times']:
    print("Tempo atteso per rovina:", np.mean(results['ruin_times']))
else:
    print("Nessuna rovina osservata.")

# 3c. Tempo atteso per successo
if results['success_times']:
    print("Tempo atteso per successo:", np.mean(results['success_times']))
else:
    print("Nessun successo osservato.")

# 4a. Verifica soluzione teorica
sim_probs = []
for _ in range(50):
    sim_probs.append(run_simulations(N, C0, C_max, p, T_max)['prob_ruin'])
fig = px.histogram(sim_probs, nbins=10, title='Distribuzione probabilità di rovina (50 simulazioni)')
fig.add_vline(x=theoretical_ruin_prob(C0, C_max, p), line_dash='dash', line_color='red')
fig.update_layout(xaxis_title='Probabilità di rovina', yaxis_title='Frequenza')
fig.show()

# 4b. C_max infinito, vari p
p_values = np.arange(0.05, 1.0, 0.1)
ruin_probs = []
for p_val in p_values:
    res = run_simulations(N, C0, float('inf'), p_val, T_max)
    ruin_probs.append(res['prob_ruin'])
fig = px.line(
    x=p_values, y=ruin_probs, markers=True,
    title='Probabilità di rovina vs p (C_max infinito)'
)
fig.update_layout(xaxis_title='p', yaxis_title='Probabilità di rovina (C_max infinito)')
fig.show()

# 4c. p > 0.5
res = run_simulations(N, C0, float('inf'), 0.6, T_max)
print("p=0.6, Probabilità di rovina:", res['prob_ruin'])

# 5a. Attesa e varianza della probabilità di rovina
K = 100
probs = [run_simulations(N, C0, C_max, p, T_max)['prob_ruin'] for _ in range(K)]
print("Attesa:", np.mean(probs))
print("Varianza:", np.var(probs))

# 5b. Intervallo di confidenza 95%
for K_val in [50, 100, 200, 2000]:
    probs = [run_simulations(N, C0, C_max, p, T_max)['prob_ruin'] for _ in range(K_val)]
    mean = np.mean(probs)
    std = np.std(probs)
    ci = norm.interval(0.95, loc=mean, scale=std/np.sqrt(K_val))
    print(f"K={K_val}: IC 95% = {ci}")

# 5c. Teorema Limite Centrale
import pandas as pd
sample_sizes = [10, 30, 50, 100]
df_means = []
for n in sample_sizes:
    means = [np.mean([run_simulations(N, C0, C_max, p, T_max)['prob_ruin'] for _ in range(n)]) for _ in range(50)]
    df_means.extend({'n': n, 'media': m} for m in means)
fig = px.histogram(pd.DataFrame(df_means), x='media', color='n', barmode='overlay',
                   title='Distribuzioni delle medie campionarie')
fig.update_layout(xaxis_title='Media campionaria', yaxis_title='Frequenza')
fig.show()

# 5d. Grafico probabilità di rovina vs p + regressione
p_vals = np.linspace(0.05, 0.95, 19)
ruin_probs = [run_simulations(N, C0, C_max, p_val, T_max)['prob_ruin'] for p_val in p_vals]
model = LinearRegression().fit(p_vals.reshape(-1,1), ruin_probs)
fig = px.scatter(x=p_vals, y=ruin_probs, title='Probabilità di rovina vs p con regressione')
fig.add_traces(px.line(x=p_vals, y=model.predict(p_vals.reshape(-1,1))).data)
fig.update_layout(xaxis_title='p', yaxis_title='Probabilità di rovina')
fig.show()
print("Coefficiente regressione:", model.coef_[0])
print("Intercept:", model.intercept_)


Probabilità di rovina: 0.891
Probabilità di successo: 0.109


T_max=100: Probabilità di rovina=0.677
T_max=500: Probabilità di rovina=0.869
T_max=1000: Probabilità di rovina=0.880
Tempo atteso per rovina: 77.80246913580247
Tempo atteso per successo: 74.45871559633028


p=0.6, Probabilità di rovina: 0.015
Attesa: 0.8817100000000001
Varianza: 8.410590000000015e-05
K=50: IC 95% = (0.879466079668669, 0.8846539203313312)
K=100: IC 95% = (0.879490972591653, 0.8836690274083475)
K=200: IC 95% = (0.8794643508402444, 0.8823156491597555)
K=2000: IC 95% = (0.8806679909248255, 0.8815460090751744)


Coefficiente regressione: -1.5669473684210533
Intercept: 1.2850000000000001
